[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/paulrobustelli/CSMP_150_2026/blob/prinz-transfer-operator/notebooks/Prinz_Potential_Transfer_Operator.ipynb)

# Transfer operators on a four-well Prinz-style potential

**CSMP 150 — worked computational lesson**

Run all cells in order in Google Colab (Python 3). Only NumPy, SciPy and Matplotlib are needed; they are included in Colab. No external files or MSM packages are required.

This notebook develops the earlier lecture code into a **reversible Metropolis-chain discretization**. All time and lag units are **Monte Carlo steps**: one application of the transition matrix, including rejected proposals. These are not calibrated molecular dynamics times.

We will connect equilibrium, eigenfunctions, relaxation, exact lag dependence and a four-state projection. This is a teaching implementation inspired by [Prinz et al., *Markov models of molecular kinetics: Generation and validation*, JCP 134, 174105 (2011)](https://doi.org/10.1063/1.3565032). It is **not a numerical reproduction of every analysis in that paper**, nor a reconstruction of its exact benchmark dynamics.

## 1. Potential and equilibrium

We retain the explicit potential from the earlier conversation:
$$V(x)=2x^8+0.8e^{-80x^2}+0.2e^{-80(x-0.5)^2}+0.5e^{-40(x+0.5)^2}.$$
This is a **Prinz-style four-well variant**; coefficients and overall energy scales vary across implementations. Do not equate its numerical spectrum with a published benchmark without matching the potential, temperature, domain and dynamics.

On an equally spaced grid in $[-1,1]$, define probability masses
$$\pi_i=\frac{\exp[-\beta V(x_i)]}{\sum_j\exp[-\beta V(x_j)]}.$$
The grid has equal weights; $\pi_i$ is probability per grid state, not a continuous density. A density plot would use $\pi_i/\Delta x$. We use $\beta=5$ to emphasize barriers; set it to 1 to recover the earlier example. Energy and position are dimensionless.

In [ ]:
import sys
import numpy as np
import scipy
from scipy.linalg import eigh
import matplotlib
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 110, "font.size": 11})
print("Python", sys.version.split()[0], "NumPy", np.__version__,
      "SciPy", scipy.__version__, "Matplotlib", matplotlib.__version__)

def potential(x):
    """Explicit lecture variant of the four-well Prinz potential."""
    return (2*x**8 + 0.8*np.exp(-80*x**2)
            + 0.2*np.exp(-80*(x-0.5)**2)
            + 0.5*np.exp(-40*(x+0.5)**2))

n = 160
beta = 5.0
proposal = 0.25  # Probability of proposing each neighboring move.
x = np.linspace(-1.0, 1.0, n)
energy = potential(x)


## 2. Specify the kinetics

For neighboring states $j=i\pm1$, propose a move with probability $q=1/4$ and accept with
$$a_{ij}=\min\{1,e^{-\beta(V_j-V_i)}\},\qquad P_{ij}=q a_{ij}.$$
All remaining probability stays at $i$:
$$P_{ii}=1-\sum_{j\ne i}P_{ij}.$$
Out-of-domain proposals are rejected (closed boundaries). The proposal is symmetric, so
$$\pi_iP_{ij}=\pi_jP_{ji}.$$
This **detailed balance** identity implies stationarity $\pi P=\pi$. The chain is connected and lazy ($P_{ii}\ge1/2$), giving a unique stationary distribution and a nonnegative spectrum. The potential alone does not determine kinetics: changing the proposal changes relaxation.

In [ ]:
def metropolis_chain(energy, beta=5.0, proposal=0.25):
    """Return equilibrium masses and a lazy, nearest-neighbor transition matrix."""
    if beta < 0 or not 0 < proposal <= 0.25:
        raise ValueError("Use beta >= 0 and 0 < proposal <= 0.25")
    weights = np.exp(-beta * (energy - energy.min()))
    pi = weights / weights.sum()
    P = np.zeros((len(energy), len(energy)))
    for i in range(len(energy)):
        for j in (i-1, i+1):
            if 0 <= j < len(energy):
                # min(0, ...) avoids overflow for downhill moves.
                P[i, j] = proposal*np.exp(min(0., -beta*(energy[j]-energy[i])))
        P[i, i] = 1.0-P[i].sum()  # Rejections and unattempted moves.
    return pi, P

pi, P = metropolis_chain(energy, beta, proposal)
np.testing.assert_allclose(P.sum(axis=1), 1, atol=1e-14)
np.testing.assert_allclose(pi @ P, pi, atol=1e-14)
np.testing.assert_allclose(pi[:, None]*P, pi[None, :]*P.T, atol=1e-14)
assert P.min() >= 0 and np.all(np.diag(P) >= 0.5-1e-14)
print("Stochasticity, stationarity and detailed balance passed.")


## 3. Which operator acts on what?

Use row probabilities and column observables throughout:
$$p(t+\tau)=p(t)P^\tau,\qquad [K_\tau f]_i=\sum_j(P^\tau)_{ij}f_j.$$
$K_\tau$ is the backward expectation (Koopman) operator. For a reversible chain, the equilibrium-relative density $r_i=p_i/\pi_i$ also evolves as $r(t+\tau)=P^\tau r(t)$ when written as a column; this is the equilibrium-weighted transfer-operator convention.

The right eigenfunctions satisfy
$$P\psi_k=\lambda_k\psi_k,\qquad \langle f,g\rangle_\pi=\sum_i\pi_i f_i g_i.$$
Detailed balance makes
$$S=D_\pi^{1/2}PD_\pi^{-1/2}=S^T.$$
Solve $Su_k=\lambda_ku_k$ and recover $\psi_k=D_\pi^{-1/2}u_k$. Then
$$\psi_k^TD_\pi\psi_l=\delta_{kl},\quad\psi_1=1,\quad\lambda_1=1.$$
The forward column-density modes are $\phi_k=D_\pi\psi_k$, satisfying $P^T\phi_k=\lambda_k\phi_k$. Eigenvector signs are arbitrary.

In [ ]:
sqrt_pi = np.sqrt(pi)
S = sqrt_pi[:, None]*P/sqrt_pi[None, :]
np.testing.assert_allclose(S, S.T, atol=1e-14)
lam, u = eigh(S)
lam, u = lam[::-1], u[:, ::-1]
psi = u/sqrt_pi[:, None]
# Fix signs for repeatable plots; weighted normalization is unchanged.
for k in range(n):
    if psi[np.argmax(np.abs(psi[:, k])), k] < 0:
        psi[:, k] *= -1
phi = pi[:, None]*psi
np.testing.assert_allclose(P @ psi, psi*lam, atol=1e-10)
np.testing.assert_allclose(P.T @ phi, phi*lam, atol=1e-12)
np.testing.assert_allclose(psi.T @ (pi[:, None]*psi), np.eye(n), atol=1e-12)
np.testing.assert_allclose(psi[:, 0], 1, atol=1e-10)
assert abs(lam[0]-1) < 1e-12 and lam[-1] >= -1e-12
assert np.all((lam[1:5] > 0) & (lam[1:5] < 1))
timescales = -1/np.log(lam[1:9])
print("mode   eigenvalue       relaxation time (Monte Carlo steps)")
for k, t in enumerate(timescales, 1):
    print(f"{k+1:4d}   {lam[k]:.9f}       {t:.2f}")


## 4. Read the spectrum and eigenfunctions

At integer lag $\tau$, $\lambda_k(\tau)=\lambda_k^\tau$. For $0<\lambda_k<1$,
$$t_k=-\frac{\tau}{\log\lambda_k(\tau)}.$$
The stationary mode has infinite relaxation time and is excluded from finite-timescale plots. For a general reversible chain negative eigenvalues can occur, indicating alternating signs; our laziness avoids that complication. Zero eigenvalues decay in one step.

Inspect how eigenfunctions distinguish combinations of wells. A four-well landscape does not by itself guarantee exactly three well-separated slow processes: inspect the spectrum and the shapes.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)
ax = axes[0, 0]
ax.plot(x, energy, color="black")
ax.set(xlabel="Position x", ylabel="Potential V(x)", title="Lecture four-well potential")
ax2 = ax.twinx()
ax2.plot(x, pi, color="tab:orange")
ax2.set_ylabel("Equilibrium probability per state", color="tab:orange")
for k in range(4):
    axes[0, 1].plot(x, psi[:, k], label=f"Mode {k+1}")
axes[0, 1].set(xlabel="Position x", ylabel="Weighted-normalized eigenfunction",
               title="Backward / relative-density eigenfunctions")
axes[0, 1].legend()
lag = 100
axes[1, 0].plot(np.arange(1, 13), lam[:12]**lag, "o-")
axes[1, 0].set(xlabel="Mode", ylabel="Eigenvalue at lag", ylim=(-0.02, 1.02),
               title=f"Lag = {lag} Monte Carlo steps")
axes[1, 1].bar(np.arange(2, 10), timescales)
axes[1, 1].set(xlabel="Mode", ylabel="Relaxation time (Monte Carlo steps)", yscale="log")
plt.show()

fig, ax = plt.subplots(figsize=(9, 4), constrained_layout=True)
for k in range(4):
    ax.plot(x, phi[:, k], label=f"Mode {k+1}")
ax.set(xlabel="Position x", ylabel="Forward density mode", title="Forward modes: pi × psi")
ax.legend()
plt.show()


## 5. Probability propagation and its spectral expansion

Completeness gives
$$P^t_{ij}=\pi_j\sum_k\lambda_k^t\psi_k(i)\psi_k(j),$$
so for any initial row distribution,
$$p_j(t)=\pi_j\left[1+\sum_{k=2}^n c_k\lambda_k^t\psi_k(j)\right],\qquad c_k=\sum_i p_i(0)\psi_k(i).$$
The full expansion is exact up to numerical error. Truncating to slow modes works better at long times and can produce negative probabilities at short times; do not clip or renormalize a truncated approximation to hide that error.

In [ ]:
p0 = np.zeros(n)
p0[np.argmin(abs(x+0.75))] = 1
coeff = p0 @ psi

def spectral_probability(t, modes=n):
    """Full or truncated spectral propagation, without clipping."""
    return pi * (psi[:, :modes] @ (coeff[:modes]*lam[:modes]**t))

# Compare independent matrix-power and full spectral calculations.
for t in (0, 1, 100, 1000):
    direct = p0 @ np.linalg.matrix_power(P, t)
    spectral = spectral_probability(t)
    np.testing.assert_allclose(spectral, direct, atol=2e-11)
    np.testing.assert_allclose(spectral.sum(), 1, atol=2e-11)
    assert spectral.min() > -2e-11

slow_time = -1/np.log(lam[1])
times = sorted(set([0, 100, int(np.ceil(slow_time)), int(np.ceil(5*slow_time))]))
fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
for t in times:
    pt = spectral_probability(t)
    if t > 0:
        axes[0].plot(x, pt, label=f"{t:,} steps")
    print(f"t={t:7d}: total variation to equilibrium = {0.5*abs(pt-pi).sum():.6f}")
axes[0].plot(x, pi, "k--", label="Equilibrium")
axes[0].set(xlabel="Position x", ylabel="Probability per state", title="Relaxation from x ≈ -0.75 (initial spike omitted)")
axes[0].legend(fontsize=8)
t = int(np.ceil(slow_time))
axes[1].plot(x, spectral_probability(t), label="All modes")
axes[1].plot(x, spectral_probability(t, 4), "--", label="First four modes")
axes[1].set(xlabel="Position x", ylabel="Probability per state", title=f"Spectral approximation at {t:,} steps")
axes[1].legend()
plt.show()


## 6. Exact lag invariance

For this fully specified chain, $P(\tau)=P^\tau$ exactly. Therefore
$$-\tau/\log(\lambda_k^\tau)=-1/\log\lambda_k.$$
Flat curves are an algebraic consistency check, **not evidence that a model fitted to molecular trajectories is valid**. A projected or estimated model need not obey this identity. Large powers can underflow for fast modes; examine the slow modes here.

In [ ]:
lags = np.array([1, 2, 5, 10, 20, 50, 100, 200])
fig, ax = plt.subplots(figsize=(8, 4), constrained_layout=True)
for k in range(1, 5):
    lag_values = lam[k]**lags
    implied = -lags/np.log(lag_values)
    np.testing.assert_allclose(implied, -1/np.log(lam[k]), rtol=1e-10)
    ax.plot(lags, implied, "o-", label=f"Mode {k+1}")
ax.set(xlabel="Lag (Monte Carlo steps)", ylabel="Implied time (Monte Carlo steps)",
       title="Exact reference chain")
ax.legend()
plt.show()


## 7. Four-state projection: discretization error and validation

Group grid states using boundaries near the three barriers. Let $H_{ia}$ indicate membership of microstate $i$ in macrostate $a$. With $\Pi_a=\sum_i\pi_iH_{ia}$, equilibrium conditional projection gives
$$C(\tau)=H^TD_\pi P^\tau H,\qquad T_{ab}(\tau)=C_{ab}(\tau)/\Pi_a.$$
This is an exact equilibrium two-time conditional probability, but treating it as a macrostate Markov chain is an approximation. In general,
$$T(2\tau)\ne T(\tau)^2.$$
Compare implied timescales and this Chapman–Kolmogorov discrepancy. There is no sampling noise here: discrepancies arise from projection. These calculations illustrate the paper's discretization and validation ideas; they do not reproduce its figures or finite-data uncertainty analyses.

In [ ]:
labels = np.digitize(x, [-0.5, 0., 0.5])
H = np.eye(4)[labels]
macro_pi = pi @ H

def projected_matrix(lag):
    """Equilibrium-conditioned transition probabilities at integer lag."""
    C = H.T @ (pi[:, None]*np.linalg.matrix_power(P, int(lag))) @ H
    return C/macro_pi[:, None]

def macro_eigenvalues(T):
    root = np.sqrt(macro_pi)
    sym = root[:, None]*T/root[None, :]
    np.testing.assert_allclose(sym, sym.T, atol=1e-12)
    return eigh(sym, eigvals_only=True)[::-1]

coarse_times = []
ck_errors = []
for lag in lags:
    T = projected_matrix(lag)
    np.testing.assert_allclose(T.sum(axis=1), 1, atol=1e-11)
    np.testing.assert_allclose(macro_pi @ T, macro_pi, atol=1e-11)
    eig = macro_eigenvalues(T)[1:]
    assert np.all((eig > 0) & (eig < 1))
    coarse_times.append(-lag/np.log(eig))
    ck_errors.append(np.max(abs(projected_matrix(2*lag)-T @ T)))
coarse_times = np.array(coarse_times)
fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
for k in range(3):
    line, = axes[0].plot(lags, coarse_times[:, k], "o-", label=f"Macro mode {k+2}")
    axes[0].axhline(-1/np.log(lam[k+1]), color=line.get_color(), linestyle="--")
axes[0].set(xlabel="Lag (Monte Carlo steps)", ylabel="Implied time (Monte Carlo steps)",
            title="Four-state projection; dashed = reference")
axes[0].legend()
axes[1].plot(lags, ck_errors, "o-")
axes[1].set(xlabel="Lag (Monte Carlo steps)", ylabel="Maximum absolute matrix discrepancy",
            title="Chapman–Kolmogorov: T(2 lag) vs T(lag)²")
plt.show()
print("All numerical checks passed.")


## Explore and discuss

1. Change $\beta$ from 5 to 1. Which eigenfunctions still look approximately constant within wells? Does the gap after the fourth eigenvalue remain convincing?
2. Halve the proposal probability. Explain why equilibrium is unchanged while timescales change. In fact $P_{q/2}=(I+P_q)/2$; predict its eigenvalues before computing them.
3. Compare full and four-mode propagation at $t=0$. Why are fast modes needed to represent a localized initial state?
4. Move the macrostate boundaries into wells. Compare implied timescales and Chapman–Kolmogorov discrepancies.
5. Increase the grid resolution. The distance moved per Monte Carlo step decreases, so raw step timescales are not a spatial convergence study of fixed physical dynamics. What rescaling and boundary treatment would a diffusion approximation need?
6. For a trajectory-based extension, estimate transition counts at each lag, examine statistical uncertainty, and compare predictions against independent observations. Flat timescale curves alone are insufficient validation.

## Scope of reproduction

Implemented: potential/equilibrium, reversible transition matrix, backward and forward modes, spectrum and relaxation times, probability propagation, spectral truncation, exact lag invariance, four-state equilibrium projection and a Chapman–Kolmogorov comparison.

Not yet reproduced: the paper's exact four-well reference model and figure parameters, systematic partition-resolution studies and theoretical error-bound comparisons, finite-trajectory estimators and uncertainty analyses, and its other numerical systems. Matching those requires a separate, source-by-source reproduction specification rather than relabeling this Metropolis chain as the published dynamics.

Reference: [Prinz et al. (2011), full paper](https://publications.imp.fu-berlin.de/944/1/11_JCP_PrinzEtAl-MSM1.pdf).